In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

# Find the cleaned dataset
possible_paths = [
    Path("data/cleaned_dataset.csv"),
    Path("../data/cleaned_dataset.csv")
]

DATASET_PATH = next(
    (path for path in possible_paths if path.exists()),
    None
)

if DATASET_PATH is None:
    raise FileNotFoundError(
        "cleaned_dataset.csv not found. Complete Day 2 first."
    )

df = pd.read_csv(DATASET_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())


In [ ]:

print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nNumeric columns:")
print(df.select_dtypes(include=np.number).columns.tolist())

print("\nDate/time columns:")
print(
    df.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()
)


In [ ]:

# Count characters in the question and system prompt
df["question_char_count"] = (
    df["test_question"].astype(str).str.len()
)

df["prompt_char_count"] = (
    df["system_prompt"].astype(str).str.len()
)

# Calculate the ratio
df["question_prompt_length_ratio"] = (
    df["question_char_count"] /
    df["prompt_char_count"].replace(0, np.nan)
)

print(df[
    [
        "test_question",
        "question_char_count",
        "prompt_char_count",
        "question_prompt_length_ratio"
    ]
].head())


In [ ]:

df["question_word_count"] = (
    df["test_question"]
    .astype(str)
    .str.split()
    .str.len()
)

print(df[
    ["test_question", "question_word_count"]
].head())


In [ ]:

new_features = [
    "question_prompt_length_ratio",
    "question_word_count"
]

print("New feature statistics:")
display(df[new_features].describe())

print("\nQuestion word count by category:")
display(
    df.groupby("category")["question_word_count"]
    .agg(["count", "mean", "min", "max"])
)


In [ ]:

# Distribution of question word counts
plt.figure(figsize=(8, 5))
sns.histplot(
    data=df,
    x="question_word_count",
    bins=10,
    kde=True
)
plt.title("Distribution of Question Word Counts")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

# Compare question word counts across categories
plt.figure(figsize=(10, 5))
sns.boxplot(
    data=df,
    x="category",
    y="question_word_count"
)
plt.title("Question Word Count by Category")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Distribution of question-to-prompt ratios
plt.figure(figsize=(8, 5))
sns.histplot(
    data=df,
    x="question_prompt_length_ratio",
    bins=10,
    kde=True
)
plt.title("Question-to-Prompt Length Ratio")
plt.xlabel("Question / Prompt Character Ratio")
plt.ylabel("Frequency")
plt.show()


In [ ]:

feature_correlation = df[
    [
        "question_prompt_length_ratio",
        "question_word_count"
    ]
].corr()

print("Correlation between engineered features:")
display(feature_correlation)

plt.figure(figsize=(6, 4))
sns.heatmap(
    feature_correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Engineered Feature Correlation")
plt.show()


In [ ]:

assert "question_prompt_length_ratio" in df.columns
assert "question_word_count" in df.columns

assert df["question_word_count"].notnull().all()
assert df["question_prompt_length_ratio"].notnull().all()

print("SUCCESS: Both engineered features created!")
print("Total rows:", len(df))
print("Total columns after feature engineering:", len(df.columns))
print("New features:", new_features)


# Week 2 – Day 3: Feature Engineering

## Objective
Create meaningful new features from existing dataset columns to improve the information available for future machine learning tasks.

## Feature 1: question_prompt_length_ratio
- Formula: Question character count / System prompt character count.
- Reason: Measures the relative size of a user's question compared with the AI instructions.
- Expected benefit: Helps describe the relative input size.

## Feature 2: question_word_count
- Formula: Number of words in the test question.
- Reason: Measures the length and amount of information in each question.
- Expected benefit: Helps distinguish short questions from detailed questions.

## Dataset Review
- Numeric column: `id` is an identifier, not a meaningful predictive feature.
- Date/time columns: No date/time columns are available.
- Text columns: `system_prompt` and `test_question` were used for feature engineering.
- New features were inspected using descriptive statistics, distributions, and correlation analysis.

## Conclusion
Two new features were successfully created from existing text columns. Their distributions and relationship were explored to understand the additional information they provide. Predictive usefulness must be evaluated later using a suitable target variable and model validation.
:::After running all cells, update the conclusion with your actual observations. Don't claim the features improve model accuracy yet: this dataset has no defined prediction target, so we can examine descriptive signal but cannot establish predictive improvement.